In [ ]:
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
import pylake

In [ ]:
model = 'geneva_dummy_extended'
mitgcm_config, ds_to_plot = open_mitgcm_ds_from_config('..//config.json', model)

In [ ]:
grid_resolution = 200
ds_to_plot['YC'] = np.arange(1, len(ds_to_plot['YC'])+1) * grid_resolution - grid_resolution/2
ds_to_plot['XC'] = np.arange(1, len(ds_to_plot['XC'])+1) * grid_resolution - grid_resolution/2
ds_to_plot['YG'] = np.arange(0, len(ds_to_plot['YG'])) * grid_resolution
ds_to_plot['XG'] = np.arange(0, len(ds_to_plot['XG'])) * grid_resolution

In [ ]:
mask = ds_to_plot.THETA.isel(time=0).values != 0

In [ ]:
plt.figure(figsize=(10,3))
plt.imshow(ds_to_plot.THETA.isel(time=-1,Z=0).where(mask[0], np.nan))
plt.gca().invert_yaxis()
plt.colorbar()

# Get temperature profile averaged over the entire lake

In [ ]:
ds_to_plot['theta_nan'] = ds_to_plot['THETA'].where(mask, np.nan)

In [ ]:
ds_to_plot['mean_temp_profile'] = ds_to_plot.theta_nan.mean(dim=['XC','YC']).compute()

In [ ]:
ds_to_plot['mean_temp_profile'].plot()

# Get buoyancy frequency N

In [ ]:
N_mean = pylake.buoyancy_freq(ds_to_plot['mean_temp_profile'].isel(time=-1).values, depth=ds_to_plot.Z.values, g=9.81)

# Get vertical displacement

In [ ]:
# Initialize v_disp with same shape as THETA
v_disp = xr.full_like(ds_to_plot.THETA, fill_value=np.nan)
theta_arr = ds_to_plot.THETA.values
for idx_z in range(ds_to_plot.sizes['Z']):
    # Reference mean profile value at this depth
    v_disp.isel()

In [ ]:
def compute_iso_displacement_slice(theta_arr, ds, idx_z):
    ref_depth = ds['Z'].isel(Z=idx_z).values
    z_arr=ds['Z'].values

    # Loop over time
    d_vert_slice=[]
    for i in range(theta_arr.shape[0]):
        ref_temp = ds['mean_temp_profile'].isel(time=i,Z=idx_z).values
        diff = np.abs(theta_arr[i] - ref_temp)

        z_closest_idx = diff.argmin(axis=0)
        z_closest = z_arr[z_closest_idx]  # (time, YC, XC)

        # Displacement relative to this reference depth
        d_vert_slice.append(np.where(mask[idx_z], z_closest - ref_depth, np.nan))

    return d_vert_slice

In [ ]:
d_vert = []
theta_arr = ds_to_plot.THETA.values
for idx_z in range(ds_to_plot.sizes['Z']):
    d_vert.append(compute_iso_displacement_slice(theta_arr, ds_to_plot, idx_z))

In [ ]:
v_disp =xr.full_like(ds_to_plot.THETA, fill_value=np.nan)

In [ ]:
arr_swapped = np.swapaxes(d_vert, 0, 1)

In [ ]:
v_disp[:] = arr_swapped

In [ ]:
v_disp.to_netcdf(r"/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/dummy_extended/analysis/vertical_displacement.nc")

In [ ]:
v_disp.isel(time=-1, Z=4).plot()

In [ ]:
ds_to_plot.THETA.isel(time=72,Z=30).plot()